# Health App Analytics

Analytics case study: cohort retention, churn exploration, and price sensitivity.

## 1. Data Loading

Load datasets and standardise date columns for time-based analysis.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

In [ ]:
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password=os.environ.get("MYSQL_PASSWORD", "ayushdas@06"),
    database="health_app"
)

In [ ]:
users = pd.read_sql("SELECT * FROM users", conn)
sessions = pd.read_sql("SELECT * FROM sessions", conn)
subs = pd.read_sql("SELECT * FROM subscriptions", conn)
payments = pd.read_sql("SELECT * FROM payments", conn)

In [ ]:
users["signup_date"] = pd.to_datetime(users["signup_date"])
sessions["date"] = pd.to_datetime(sessions["date"])
subs["start_date"] = pd.to_datetime(subs["start_date"])
payments["date"] = pd.to_datetime(payments["date"])

## 2. Cohort Retention Analysis

Group users by signup month (cohort), then measure what % are still active in later months.

In [ ]:
users["cohort_month"] = users["signup_date"].dt.to_period("M")
sessions = sessions.merge(users[["user_id", "cohort_month"]], on="user_id", how="left")
sessions["activity_month"] = sessions["date"].dt.to_period("M")

In [ ]:
cohort_users = sessions.groupby(["cohort_month", "activity_month"])["user_id"].nunique().reset_index()
cohort_pivot = cohort_users.pivot(index="cohort_month", columns="activity_month", values="user_id")

In [ ]:
cohort_sizes = users.groupby("cohort_month").size()
retention_pct = cohort_pivot.div(cohort_sizes, axis=0) * 100
retention_pct = retention_pct.round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(retention_pct.values, aspect="auto", cmap="YlGnBu", vmin=0, vmax=100)
ax.set_xticks(range(len(retention_pct.columns)))
ax.set_xticklabels([str(c) for c in cohort_pivot.columns], rotation=45, ha="right")
ax.set_yticks(range(len(cohort_pivot.index)))
ax.set_yticklabels([str(i) for i in cohort_pivot.index])
ax.set_xlabel("Activity month")
ax.set_ylabel("Cohort (signup month)")
plt.colorbar(im, ax=ax, label="Retention %")
plt.title("Cohort retention: % of cohort still active by month")
plt.tight_layout()
plt.show()

## 3. Churn Exploration

Compare engagement (sessions) between users who renewed vs churned to spot behaviour differences.

In [ ]:

subs["churn_flag"] = (subs["renewal_status"] == "churned")
session_count = sessions.groupby("user_id").size().reset_index(name="session_count")
subs_with_sessions = subs.merge(session_count, on="user_id", how="left")
subs_with_sessions["session_count"] = subs_with_sessions["session_count"].fillna(0).astype(int)

In [ ]:
churn_insight = subs_with_sessions.groupby("churn_flag").agg(
    users=("user_id", "count"),
    avg_sessions=("session_count", "mean"),
).round(2)
churn_insight.index = ["Renewed", "Churned"]
print("Churn insight: average sessions per user by renewal status")
print(churn_insight)

## 4. Price Sensitivity Simulation

Simulate revenue at two price points: lower price (₹199) typically converts more users; higher (₹299) converts fewer but earns more per user.

In [ ]:
n_potential = 10_000
conv_199 = 0.08   
conv_299 = 0.05   
price_199 = 199
price_299 = 299

revenue_199 = n_potential * conv_199 * price_199
revenue_299 = n_potential * conv_299 * price_299

revenue_comparison = pd.DataFrame({
    "Plan": ["₹199", "₹299"],
    "Conversion_rate": [f"{conv_199*100:.0f}%", f"{conv_299*100:.0f}%"],
    "Converted_users": [int(n_potential * conv_199), int(n_potential * conv_299)],
    "Revenue_₹": [revenue_199, revenue_299],
})
print("Revenue comparison (simulated from 10,000 potential users)")
print(revenue_comparison.to_string(index=False))